In [6]:
# using this 

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import html

# --- STEP 1: THE SCRAPER ---
class MedicalDatasetBuilder:
    def __init__(self):
        self.headers = {'User-Agent': 'HealthResearchBot/1.0'}
        self.data = []
        
        self.targets = {
            "kanker.nl": {"cat": "Oncology", "sel": "main", "urls": ["https://www.kanker.nl/bibliotheek", "https://www.kanker.nl/gevolgen-van-kanker"]},
            "thuisarts.nl": {"cat": "General Medical", "sel": "article", "urls": ["https://www.thuisarts.nl/onderwerpen"]},
            "iknl.nl": {"cat": "Research", "sel": ".content-block", "urls": ["https://iknl.nl/nieuws", "https://iknl.nl/over-iknl"]},
            "nfk.nl": {"cat": "Patient Fed", "sel": ".content", "urls": ["https://nfk.nl/over-kanker"]},
            "zorgwijzer.nl": {"cat": "Policy/Insurance", "sel": ".article-content", "urls": ["https://www.zorgwijzer.nl/zorgverzekering-2026"]},
            "zorginstituutnederland.nl": {"cat": "Policy", "sel": "#content", "urls": ["https://www.zorginstituutnederland.nl/over-ons"]}
        }

    def scrape_all(self):
        for domain, info in self.targets.items():
            for url in info['urls']:
                try:
                    res = requests.get(url, headers=self.headers, timeout=10)
                    soup = BeautifulSoup(res.text, 'html.parser')
                    title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Title"
                    content_node = soup.select_one(info['sel'])
                    content = content_node.get_text(separator=' ', strip=True) if content_node else ""

                    self.data.append({
                        "Domain": domain,
                        "Category": info['cat'],
                        "Title": title,
                        "Body": content,
                        "URL": url
                    })
                    print(f"Scraped: {domain} | {title[:30]}...")
                    time.sleep(1)
                except Exception as e:
                    print(f"Error {domain}: {e}")

    def get_dataframe(self):
        return pd.DataFrame(self.data)

# --- STEP 2: THE AI CLEANER ---
class MedicalDataCleaner:
    def __init__(self, raw_df):
        self.df = raw_df.copy()
        
    def clean_text(self, text):
        if not isinstance(text, str): return ""
        text = html.unescape(text)
        text = re.sub(r'http\S+|[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        boilerplate = ["alle rechten voorbehouden", "cookie-instellingen", "lees meer"]
        for phrase in boilerplate:
            text = text.replace(phrase, "")
        return text

    def chunk_text(self, text, size=800):
        words = text.split()
        return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

    def run_pipeline(self):
        self.df['Body_Clean'] = self.df['Body'].apply(self.clean_text)
        self.df['Title_Clean'] = self.df['Title'].apply(self.clean_text)
        
        kb_table = self.df[['Domain', 'Category', 'Title_Clean', 'Body_Clean', 'URL']]
        meta_table = self.df[['Domain', 'Category']].drop_duplicates().reset_index(drop=True)
        meta_table['Trust_Level'] = "Official/Verified"
        
        chunks = []
        for _, row in self.df.iterrows():
            text_chunks = self.chunk_text(row['Body_Clean'])
            for i, chunk in enumerate(text_chunks):
                chunks.append({
                    "chunk_id": f"{row['Domain']}_{i}",
                    "parent_url": row['URL'],
                    "text": chunk,
                    "category": row['Category']
                })
        chunks_table = pd.DataFrame(chunks)
        
        return kb_table, meta_table, chunks_table

# --- STEP 3: EXECUTION ---

# 1. Scrape
builder = MedicalDatasetBuilder()
builder.scrape_all()
raw_df = builder.get_dataframe()

# 2. Clean
cleaner = MedicalDataCleaner(raw_df)
df_kb, df_meta, df_chunks = cleaner.run_pipeline()

# 3. View All Data
# Setting pandas to show all text content without clipping
pd.set_option('display.max_colwidth', None)

print("\n" + "="*50)
print("TABLE 1: KNOWLEDGE BASE (All Rows)")
print("="*50)
print(df_kb)

print("\n" + "="*50)
print("TABLE 2: METADATA & SOURCE TRUST")
print("="*50)
print(df_meta)

print("\n" + "="*50)
print("TABLE 3: CHUNKS FOR LLM (First 5 Rows)")
print("="*50)
print(df_chunks.head())

# Optional: Save to CSVs to inspect locally
# df_kb.to_csv("clean_medical_kb.csv", index=False)

Scraped: kanker.nl | Betrouwbare informatie...
Scraped: kanker.nl | Gevolgen van kanker...
Scraped: thuisarts.nl | Deze pagina bestaat niet meer...
Scraped: iknl.nl | Nieuws...
Scraped: iknl.nl | Over IKNL...
Scraped: nfk.nl | 404...
Scraped: zorgwijzer.nl | Zorgverzekering2026 vergelijke...
Scraped: zorginstituutnederland.nl | Wat wij doen...

TABLE 1: KNOWLEDGE BASE (All Rows)
                      Domain          Category  \
0                  kanker.nl          Oncology   
1                  kanker.nl          Oncology   
2               thuisarts.nl   General Medical   
3                    iknl.nl          Research   
4                    iknl.nl          Research   
5                     nfk.nl       Patient Fed   
6              zorgwijzer.nl  Policy/Insurance   
7  zorginstituutnederland.nl            Policy   

                       Title_Clean  \
0           Betrouwbare informatie   
1              Gevolgen van kanker   
2    Deze pagina bestaat niet meer   
3              

In [7]:
# Create a specialized table for CSN Question Analysis
csn_analysis_df = df_kb.copy()

# 1. Action Tagging
def auto_tag(text):
    tags = []
    if "wachttijd" in text.lower(): tags.append("Waiting Times")
    if "verwijzing" in text.lower(): tags.append("Referrals")
    if "second opinion" in text.lower(): tags.append("Second Opinion")
    return ", ".join(tags)

csn_analysis_df['Action_Tags'] = csn_analysis_df['Body_Clean'].apply(auto_tag)

# 2. Stakeholder Identification
csn_analysis_df['Stakeholder'] = csn_analysis_df['Body_Clean'].apply(
    lambda x: "Caregiver" if "naaste" in x.lower() or "partner" in x.lower() else "Patient"
)

# 3. Refresh Tracking
csn_analysis_df['Scrape_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d')

In [8]:
import pandas as pd
import IPython.display as display # Use this if you are in a Jupyter Notebook

class DatasetVisualizer:
    @staticmethod
    def show_all(df_kb, df_meta, df_chunks):
        # 1. Terminal/Notebook Formatting
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_rows', 10)  # Shows top/bottom 5 to keep it clean
        pd.set_option('display.max_colwidth', 50) # Keeps body text from taking up the whole screen

        print("\n" + "█"*60)
        print(" VIEWING: KNOWLEDGE BASE (Primary Journey Data)")
        print("█"*60)
        print(df_kb[['Category', 'Journey_Stage', 'Action_Tags', 'Title_Clean']].head(10))

        print("\n" + "█"*60)
        print(" VIEWING: SOURCE FRESHNESS METADATA")
        print("█"*60)
        print(df_meta)

    @staticmethod
    def export_for_audit(df_kb):
        """Saves a CSV that you can open in Excel to manually check for gaps."""
        filename = "navigator_audit_log.csv"
        df_kb.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n✅ Audit dataset saved to: {filename}")

# --- ENHANCED CLEANER WITH JOURNEY TAGS ---
# --- ENHANCED CLEANER WITH JOURNEY TAGS ---
class StrategicCleaner(MedicalDataCleaner):
    def run_strategic_pipeline(self):
        # Trigger the base cleaning first
        kb_raw, meta, chunks = self.run_pipeline()
        
        # FIX: Create an explicit copy so we don't trigger the SettingWithCopyWarning
        kb = kb_raw.copy()
        
        # Adding Journey Stages based on keywords in Body
        def get_journey_stage(text):
            text = text.lower()
            if any(w in text for w in ["onderzoek", "uitslag", "biopt"]): return "Diagnosis"
            if any(w in text for w in ["bestraling", "chemo", "operatie"]): return "Treatment"
            if any(w in text for w in ["nazorg", "herstel", "controle"]): return "Aftercare"
            return "General Information"

        def get_action_tag(text):
            text = text.lower()
            tags = []
            if "wachttijd" in text: tags.append("Waiting Times")
            if "verwijzing" in text: tags.append("Referrals")
            if "second opinion" in text: tags.append("Decision Support")
            return ", ".join(tags) if tags else "General"

        # Now we can safely set these columns using .loc
        kb.loc[:, 'Journey_Stage'] = kb['Body_Clean'].apply(get_journey_stage)
        kb.loc[:, 'Action_Tags'] = kb['Body_Clean'].apply(get_action_tag)
        
        return kb, meta, chunks

# --- EXECUTION ---
# (Using your previous builder/scrape results)
cleaner = StrategicCleaner(raw_df)
df_kb, df_meta, df_chunks = cleaner.run_strategic_pipeline()

# Visualizing
viz = DatasetVisualizer()
viz.show_all(df_kb, df_meta, df_chunks)
viz.export_for_audit(df_kb)


████████████████████████████████████████████████████████████
 VIEWING: KNOWLEDGE BASE (Primary Journey Data)
████████████████████████████████████████████████████████████
           Category        Journey_Stage Action_Tags  \
0          Oncology            Diagnosis     General   
1          Oncology  General Information     General   
2   General Medical  General Information     General   
3          Research  General Information     General   
4          Research  General Information     General   
5       Patient Fed  General Information     General   
6  Policy/Insurance  General Information     General   
7            Policy  General Information     General   

                       Title_Clean  
0           Betrouwbare informatie  
1              Gevolgen van kanker  
2    Deze pagina bestaat niet meer  
3                           Nieuws  
4                        Over IKNL  
5                              404  
6  Zorgverzekering2026 vergelijken  
7                     Wat wi

In [ ]:
# This creates a scrollable, color-coded view directly 
df_kb.head(20).style.set_properties(**{'background-color': '#f9f9f9', 'border-color': 'black'})

,Domain,Category,Title_Clean,Body_Clean,URL,Journey_Stage,Action_Tags
0,kanker.nl,Oncology,Betrouwbare informatie,"A Alvleesklierkanker Anuskanker Atypisch fibroxanthoom B Baarmoederhalskanker Baarmoederkanker Baarmoedersarcoom Basaalcelcarcinoom Bestraling Bijnierkanker Blaaskanker Bloedtransfusie Borstkanker Borstkanker bij mannen Botkanker (botsarcoom) Buikvlieskanker C Cannabis bij kanker Chemotherapie Cijfers over kanker Cognitieve problemen bij kanker Complementaire zorg (aanvullende zorg) bij kanker D Darmkanker (dikkedarmkanker) DNA-onderzoek bij mensen met kanker Doelgerichte therapie Dunnedarmkanker E Eerder in de overgang door kanker Eierstokkanker Endeldarmkanker Erfelijkheid en kanker F Financiën en kanker G Galblaaskanker Galwegkanker Gastro-intestinale stromale tumor Gezond leven met en na kanker H Hart- en vaatziekten door de behandeling van kanker Hartkanker Hersentumoren Hodgkinlymfoom Hoge urinewegkanker Hoofd-halskanker Hormoontherapie (anti-hormonale therapie) Huidkanker Huidlymfoom Hyperbare zuurstoftherapie Hyperthermie I Immunotherapie In gesprek met je arts J Je keuzes en rechten als kankerpatiënt Jong & kanker K Kanker bij kinderen Kanker en andere ziektes hebben Kanker en je gezin Kanker, wat nu? Keelkanker L Leukemie Leverkanker Lipkanker Longkanker Luchtpijpkanker Lymfeklierkanker Lymfoedeem bij kanker M Maagkanker Melanoom Merkelcelcarcinoom Mesothelioom Mondkanker MPN (myeloproliferatieve neoplasmata) Mucosaal melanoom Multipel myeloom (ziekte van Kahler) Myelodysplastisch syndroom N Neuro-endocriene tumoren Neuropathie bij kanker Neuskanker Nierkanker Niet meer beter worden Non-hodgkinlymfoom O Onderzoeken bij kanker Oogkanker Oogmelanoom Oorkanker Opereren zonder snijden (interventieradiologie) P Papil van Vater-kanker Peniskanker Pijn bij kanker Plasbuiskanker Plaveiselcelcarcinoom Primaire tumor onbekend (PTO) Problemen met mond en gebit door kanker Prostaatkanker Protonentherapie Pseudomyxoma peritonei (PMP) S Schaamlipkanker (vulvakanker) Schildklierkanker Seksualiteit bij kanker Slokdarmkanker Speekselklierkanker Stamceltransplantatie Strottenhoofdkanker Studies bij kanker (trials) T Talgkliercarcinoom Thymuskanker Tongkanker Trofoblastziekten U Urachuskanker V Vaginakanker Veranderingen aan je lichaam en uiterlijk door kanker Verder met je leven na kanker Vermoeidheid bij kanker Verzekeringen en kanker Voor naasten Vruchtbaarheid en kanker W Wat is kanker? Wekedelentumoren / wekedelensarcomen Werk en kanker Z Zaadbalkanker (teelbalkanker) Zeldzame kankers Ziekte van Waldenström Zwanger en kanker",https://www.kanker.nl/bibliotheek,Diagnosis,General
1,kanker.nl,Oncology,Gevolgen van kanker,"C Cognitieve problemen bij kanker Complementaire zorg (aanvullende zorg) bij kanker E Eerder in de overgang door kanker F Financiën en kanker H Hart- en vaatziekten door de behandeling van kanker K Kanker, wat nu? L Lymfoedeem bij kanker N Neuropathie bij kanker Niet meer beter worden P Pijn bij kanker Problemen met mond en gebit door kanker S Seksualiteit bij kanker V Veranderingen aan je lichaam en uiterlijk door kanker Verder met je leven na kanker Vermoeidheid bij kanker Verzekeringen en kanker Vruchtbaarheid en kanker W Werk en kanker",https://www.kanker.nl/gevolgen-van-kanker,General Information,General
2,thuisarts.nl,General Medical,Deze pagina bestaat niet meer,,https://www.thuisarts.nl/onderwerpen,General Information,General
3,iknl.nl,Research,Nieuws,,https://iknl.nl/nieuws,General Information,General
4,iknl.nl,Research,Over IKNL,,https://iknl.nl/over-iknl,General Information,General
5,nfk.nl,Patient Fed,404,,https://nfk.nl/over-kanker,General Information,General
6,zorgwijzer.nl,Policy/Insurance,Zorgverzekering2026 vergelijken,,https://www.zorgwijzer.nl/zorgverzekering-2026,General Information,General
7,zorginstituutnederland.nl,Policy,Wat wij doen,,https://www.zorginstituutnederland.nl/over-ons,General Information,General


In [10]:

# This script is great
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.kanker.nl"
START_URL = "https://www.kanker.nl/kankersoorten"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_cancer_links():
    """Extracts all specific unique cancer links using the main AZ index container."""
    print(f"Fetching overview page: {START_URL}")
    response = requests.get(START_URL, headers=HEADERS)
    if response.status_code != 200:
        print(f"Failed to load overview page. Status code: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    cancer_links = {}

    # Target the primary layout grid list where alphabetical diseases are housed
    grid_container = soup.find('div', class_='dynamic-grid') or soup.find('main')
    
    if not grid_container:
        grid_container = soup # fallback if structure alters

    for link in grid_container.find_all('a', href=True):
        href = link['href']
        # Extract links formatted like /kankersoorten/alvleesklierkanker
        if href.startswith('/kankersoorten/') and len(href.split('/')) == 3:
            name = link.get_text(strip=True)
            full_url = BASE_URL + href
            
            # Avoid picking up generic tracking shortcuts or duplicate labels
            if name and "bekijk alle" not in name.lower() and "meer kankersoorten" not in name.lower():
                if name not in cancer_links:
                    cancer_links[name] = full_url

    print(f"Found {len(cancer_links)} unique cancer variants to process.")
    return cancer_links

def scrape_cancer_details(cancer_name, hub_url):
    """Navigates to subpages and extracts structural parameters securely."""
    slug = hub_url.split('/')[-1]
    target_url = f"{hub_url}/algemeen/wat-is-{slug}"
    
    print(f"Scraping detailed info for [{cancer_name}] via {target_url}...")
    
    data_row = {
        "Cancer Type": cancer_name,
        "Source URL": target_url,
        "General Description": "N/A",
        "Prognosis (Vooruitzichten)": "N/A",
        "Symptoms (Symptomen)": "N/A",
        "Causes (Oorzaken)": "N/A",
        "Treatments (Behandeling)": "N/A"
    }
    
    response = requests.get(target_url, headers=HEADERS)
    if response.status_code != 200:
        target_url = hub_url
        response = requests.get(target_url, headers=HEADERS)
        if response.status_code != 200:
            return data_row

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # --- FIXED GENERAL DESCRIPTION EXTRACTION WITH SYSTEM NOISE FILTER ---
    main_article = soup.find('article') or soup.find('main') or soup
    
    intro_paragraphs = []
    skip_phrases = [
        "deze informatie is gecontroleerd", 
        "naar colofon", 
        "printen", 
        "opslaan", 
        "delen",
        "luister"
    ]
    
    for p in main_article.find_all('p'):
        p_text = p.get_text(strip=True)
        if not p_text:
            continue
        if "lees op deze pagina" in p_text.lower():
            break
        if any(phrase in p_text.lower() for phrase in skip_phrases):
            continue
        intro_paragraphs.append(p_text)
        
    if intro_paragraphs:
        data_row["General Description"] = intro_paragraphs[0]

    # --- SUBSECTIONS HEADER MAPPING ---
    headings = soup.find_all(['h2', 'h3'])
    for heading in headings:
        heading_text = heading.get_text(strip=True).lower()
        
        content_paragraphs = []
        next_node = heading.next_sibling
        while next_node:
            if next_node.name in ['h2', 'h3']:
                break
            if next_node.name in ['p', 'ul', 'ol']:
                content_paragraphs.append(next_node.get_text(" ", strip=True))
            next_node = next_node.next_sibling
            
        section_content = " ".join(content_paragraphs)
        if not section_content:
            continue
            
        if 'vooruitzichten' in heading_text:
            data_row["Prognosis (Vooruitzichten)"] = section_content
        elif 'symptomen' in heading_text or 'klachten' in heading_text:
            data_row["Symptoms (Symptomen)"] = section_content
        elif 'oorzaken' in heading_text or 'risicofactoren' in heading_text:
            data_row["Causes (Oorzaken)"] = section_content
        elif 'behandeling' in heading_text:
            data_row["Treatments (Behandeling)"] = section_content

    return data_row

def main():
    cancer_directory = get_cancer_links()
    if not cancer_directory:
        print("No paths discovered. Check structural layout rules.")
        return

    all_cancer_data = []
    
    # Process items inside the clean directory mapping
    for name, url in cancer_directory.items():
        try:
            row_data = scrape_cancer_details(name, url)
            all_cancer_data.append(row_data)
        except Exception as e:
            print(f"Error scraping {name}: {e}")
            
        time.sleep(1.5)  # Throttling protection

    # Save to table files
    df = pd.DataFrame(all_cancer_data)
    df.to_csv("kanker_nl_master_table.csv", index=False, encoding='utf-8-sig')
    df.to_excel("kanker_nl_master_table.xlsx", index=False)
    print("\nMaster table files generated successfully with correct isolated links!")

if __name__ == "__main__":
    main()

Fetching overview page: https://www.kanker.nl/kankersoorten
Found 73 unique cancer variants to process.
Scraping detailed info for [Alvleesklierkanker] via https://www.kanker.nl/kankersoorten/alvleesklierkanker/algemeen/wat-is-alvleesklierkanker...
Scraping detailed info for [Anuskanker] via https://www.kanker.nl/kankersoorten/anuskanker/algemeen/wat-is-anuskanker...
Scraping detailed info for [Atypisch fibroxanthoom] via https://www.kanker.nl/kankersoorten/atypisch-fibroxanthoom/algemeen/wat-is-atypisch-fibroxanthoom...
Scraping detailed info for [Baarmoederhalskanker] via https://www.kanker.nl/kankersoorten/baarmoederhalskanker/algemeen/wat-is-baarmoederhalskanker...
Scraping detailed info for [Baarmoederkanker] via https://www.kanker.nl/kankersoorten/baarmoederkanker/algemeen/wat-is-baarmoederkanker...
Scraping detailed info for [Baarmoedersarcoom] via https://www.kanker.nl/kankersoorten/baarmoedersarcoom/algemeen/wat-is-baarmoedersarcoom...
Scraping detailed info for [Basaalcelcarc